In [1]:
import numpy as np
import numpy.random as rnd


In [2]:
class Detector:
    def __init__(self, R=2.0, N=40, r=0.1, a=0.5):
        if R <= 0 or r <= 0:
            raise ValueError("Радиусы R и r должны быть положительными.")
        if not isinstance(N, (int, np.integer)) or N < 1:
            raise ValueError("Число ФЭУ N должно быть положительным целым.")
        if a < 0:
            raise ValueError("Среднее число фотоэлектронов a не может быть отрицательным.")

        self.R = R
        self.N = N
        self.r = r
        self.a = a

        self.fmt_positions = self.generate_fmts()

        diff = self.fmt_positions[:, None, :] - self.fmt_positions[None, :, :]
        distances = np.linalg.norm(diff, axis=2)
        np.fill_diagonal(distances, np.inf)
        if np.min(distances) < 2 * self.r:
            raise ValueError(
                "Расстояние между центрами ФЭУ меньше 2r. Уменьшите r или N."
            )

    def generate_fmts(self):
        """Разместим центры ФЭУ по сфере методом Фибоначчи."""
        i = np.arange(self.N)
        cos_theta = 1 - 2 * (i + 0.5) / self.N
        sin_theta = np.sqrt(1 - cos_theta**2)
        golden_angle = np.pi * (3 - np.sqrt(5))
        phi = i * golden_angle

        x = self.R * sin_theta * np.cos(phi)
        y = self.R * sin_theta * np.sin(phi)
        z = self.R * cos_theta
        return np.column_stack([x, y, z])

    def find_fmt(self, point):
        """
        Проверим, попадает ли точка на поверхности детектора на ФЭУ.
        Радиус области ФЭУ задается расстоянием по хорде до его центра.
        Вернем номер ФЭУ (0..N-1) или -1, если мимо.
        """
        diff = self.fmt_positions - point
        dist = np.linalg.norm(diff, axis=1)
        best_fmt = np.argmin(dist)
        if dist[best_fmt] <= self.r:
            return int(best_fmt)
        return -1

    def get_intersection(self, pos, direction):

        pos = np.asarray(pos, dtype=float)
        direction = np.asarray(direction, dtype=float)
        direction_norm = np.linalg.norm(direction)
        if direction_norm == 0:
            raise ValueError("Направление фотона не может быть нулевым.")
        direction = direction / direction_norm

        b = np.dot(pos, direction)
        c = np.dot(pos, pos) - self.R**2
        if c > 0:
            raise ValueError("Источник должен находиться внутри детектора.")
        if c == 0:
            return pos.copy()  # На границе фотон сразу поглощается.
        L = -b + np.sqrt(b**2 - c)
        return pos + L * direction

    def generate_fe(self):

        return rnd.poisson(self.a)


In [3]:
class Source:
    def __init__(self, Y=10000, pos=(0.0, 0.0, 0.0)):
        self.Y = Y
        self.pos = np.array(pos, dtype=float)

    def generate_directions(self, E):

        n_photons = int(self.Y * E)
        phi = rnd.uniform(0, 2 * np.pi, n_photons)
        cos_theta = rnd.uniform(-1, 1, n_photons)
        sin_theta = np.sqrt(1 - cos_theta**2)

        vx = sin_theta * np.cos(phi)
        vy = sin_theta * np.sin(phi)
        vz = cos_theta
        return np.column_stack([vx, vy, vz])


In [4]:
def simulate_event(det, source, E):
    """Смоделируем одно событие без рассеяния и поглощения в объеме."""
    directions = source.generate_directions(E)
    fmt_signals = np.zeros(det.N, dtype=int)
    photons_count = 0

    for direct in directions:
        pt = det.get_intersection(source.pos, direct)
        fmt_id = det.find_fmt(pt)
        if fmt_id != -1:
            photons_count += 1
            fmt_signals[fmt_id] += det.generate_fe()
        # Достигнув границы, фотон поглощается независимо от попадания на ФЭУ.

    return fmt_signals, photons_count, len(directions)


In [5]:
# Фиксируем seed для воспроизводимого результата одного события.
seed = 42
rnd.seed(seed)

det = Detector(R=2.0, N=40, r=0.1, a=0.5)
source = Source(Y=10000, pos=(0.0, 0.0, 0.0))
E_event = 1.0

fmt_signals, photons_count, n_photons = simulate_event(det, source, E_event)
Q_total = int(np.sum(fmt_signals))

p = det.N * det.r**2 / (4 * det.R**2)
Q_ozhid = det.a * n_photons * p

sigma_Q = np.sqrt(n_photons * (p * det.a + p * (1 - p) * det.a**2))

print(f"Начальное состояние генератора (seed): {seed}")
print(f"Энергия события: {E_event:.1f} МэВ")
print(f"Всего излучено фотонов: {n_photons}")
print(f"Попало на ФЭУ фотонов: {photons_count}")
print(f"Ожидаемое число попаданий: {n_photons * p:.2f}")
print(f"Полный собранный заряд Q: {Q_total} фотоэлектронов")
print(f"Ожидаемый средний заряд: {Q_ozhid:.2f} фотоэлектронов")
print(f"Стандартное отклонение заряда: {sigma_Q:.2f} фотоэлектронов")
if sigma_Q > 0:
    print(f"Отклонение от среднего: {(Q_total - Q_ozhid) / sigma_Q:.2f} sigma")


Начальное состояние генератора (seed): 42
Энергия события: 1.0 МэВ
Всего излучено фотонов: 10000
Попало на ФЭУ фотонов: 239
Ожидаемое число попаданий: 250.00
Полный собранный заряд Q: 126 фотоэлектронов
Ожидаемый средний заряд: 125.00 фотоэлектронов
Стандартное отклонение заряда: 13.64 фотоэлектронов
Отклонение от среднего: 0.07 sigma
